# GHIM Energy Module — Energy Choice Model: Design Discussion

**Purpose**: This notebook documents the full mathematical framework for energy choice in
the GHIM model, from GDP to technology shares, and discusses two planned improvements:

1. **Time-varying preference factors** — wire the existing `PREF_DECAY_RATE` and make it technology-specific
2. **Asymmetric stock turnover** — separate grow/decline $\tau$ per technology

This is a *design document* for discussion. Each section is a separate cell so you can
leave inline comments between them.

---
## Layer 1: GDP → Total Energy Demand

The KLEM driver computes aggregate energy demand as a function of GDP and a composite energy price index:

$$E = E_{\text{base}} \cdot \frac{Y}{Y_{\text{base}}} \cdot \left(\frac{\bar{P}}{\bar{P}_{\text{base}}}\right)^{-\sigma_{EM}}$$

where:

| Symbol | Meaning | Current value |
|--------|---------|--------------|
| $E_{\text{base}}$ | Base-year total final energy demand (EJ) | Region-specific |
| $Y$ | Gross output from DICE production function (billion USD PPP) | Endogenous |
| $\bar{P}$ | Composite energy price index (simple average across carriers) | Endogenous |
| $\sigma_{EM}$ | Energy-materials substitution elasticity | 0.5 |

GDP itself is endogenous:

$$Y = A(t) \cdot K(t)^{\alpha} \cdot L(t)^{1-\alpha}$$

with $\alpha = 0.3$, and TFP $A(t)$ pre-calibrated from SSP GDP paths.

### CHT: Here what does base year mean?

* Is it fixed 2020? or
* Previous model year?

The latter seems more suitable for `recursive` dynamic model.

#### Response: Base Year Is Fixed 2020 — Here's Why, and When Rolling-Base Would Help

**Current behavior**: Fixed 2020. In `KLEMDriver.__init__()`, `base_gdp` and `base_energy` are set once from base-year data and never updated. Similarly, `FinalDemand._base_gdp` is set once during `calibrate()`. Every projection period computes demand as a ratio to 2020 values:

$$E_s = E_{s,2020} \cdot \left(\frac{Y(t)}{Y_{2020}}\right)^{\varepsilon_s}$$

**Why fixed base works (and why it's the common IAM choice)**:

TFP $A(t)$ is calibrated against the same fixed base — if you re-base demand each period but not TFP, the two diverge and the energy-GDP feedback becomes inconsistent. The income elasticity formula is well-defined only with a consistent $Y_0$. Re-basing each period would turn it into a growth-rate model:

$$E(t+\Delta t) = E(t) \cdot \left(\frac{Y(t+\Delta t)}{Y(t)}\right)^{\varepsilon}$$

This compounds elasticity errors period-by-period — a small calibration bias in $\varepsilon$ grows exponentially over the projection horizon.

GCAM, WITCH, and REMIND all use fixed base-year calibration for this reason.

**However, CHT's point has merit for recursive-dynamic modeling**:

A rolling-base approach would better represent structural breaks. For example, if a region industrializes rapidly between 2020–2040, the 2020 demand structure is no longer a meaningful reference for 2050 demand. A fixed base can't capture post-transition demand patterns.

**Proposed hybrid** (Phase 3 consideration): Keep fixed 2020 for calibration and preference factors (these must be anchored), but allow optional **re-anchoring** at user-specified years for demand scaling. E.g., re-anchor at 2040 after a structural break, resetting $E_{s,\text{base}}$ and $Y_{\text{base}}$ to observed/simulated 2040 values. This preserves consistency while allowing the model to "forget" outdated base-year structure when appropriate.

## Layer 2: Sector Demand Scaling

Total energy is split across three demand sectors (industry, buildings, transport), each
scaling independently with GDP via sector-specific income elasticities:

$$E_s = E_{s,\text{base}} \cdot \left(\frac{Y}{Y_{\text{base}}}\right)^{\varepsilon_s}$$

| Sector | $\varepsilon_s$ |
|--------|----------------|
| Industry | 0.6 |
| Buildings | 0.5 |
| Transport | 0.7 |

**Note on Layer 1 vs Layer 2**: These two layers are currently *independent* — Layer 1
computes total energy for the KLEM price feedback loop, while Layer 2 scales sector demands
separately via `FinalDemand.compute_demand()`. The two sums are not explicitly reconciled.
This is a known simplification; in a more sophisticated model (WITCH, REMIND), total energy
would emerge from the CES nesting directly.

### CHT: I think those two layers should be reconciled. Can you suggest ways to reconcile them with not so much solving burden?

#### Response: Three Ways to Reconcile Layer 1 and Layer 2

The problem: Layer 1 gives total energy $E_{\text{total}}$ from KLEM, Layer 2 gives sector demands $E_s$ from income elasticities, and $\sum_s E_s \neq E_{\text{total}}$ in general. Three options, ordered by implementation effort:

---

**Option 1: Proportional scaling** (recommended for immediate implementation)

After computing sector demands $E_s$ from Layer 2, scale them so they match the Layer 1 aggregate:

$$E_s^{\text{scaled}} = E_s \cdot \frac{E_{\text{total}}}{\sum_{s'} E_{s'}}$$

This is essentially one line of code:
```python
scale = total_energy / sum(sector_demands.values())
sector_demands = {s: e * scale for s, e in sector_demands.items()}
```

**Pros**: Zero risk, trivial to implement, preserves relative sector shares from income elasticities while matching the KLEM aggregate. The price feedback loop through $\sigma_{EM}$ is preserved.

**Cons**: The scaling factor is a fudge — it has no economic interpretation. If Layer 1 and Layer 2 diverge significantly (e.g., after 2080 when income elasticities compound), the scaling factor becomes large and the sector-level demands are effectively overridden by Layer 1.

---

**Option 2: Drop Layer 1, derive aggregate from Layer 2** (clean long-term target)

Let total energy emerge bottom-up: $E_{\text{total}} = \sum_s E_s$. Remove `compute_energy_demand()` from the KLEM driver. The price feedback uses the demand-weighted average carrier price instead of the current simple average:

$$\bar{P} = \frac{\sum_s E_s \cdot \bar{P}_s}{\sum_s E_s}$$

**Pros**: Conceptually cleaner — no reconciliation needed because there's only one demand calculation. Sector-specific price elasticities emerge naturally from the carrier mix.

**Cons**: Loses the aggregate substitution elasticity $\sigma_{EM}$ between energy and materials in the KLEM function. This elasticity captures the economy's ability to substitute away from energy *in aggregate* (e.g., shifting GDP composition toward services). Removing it makes total energy demand purely income-driven with no aggregate price response — only carrier-level price response through the logit.

---

**Option 3: CES bridge** (most principled, moderate effort)

Replace the independent Layer 1 with a CES aggregator that makes total energy a *function* of sector demands:

$$E_{\text{total}} = \left[\sum_s \alpha_s E_s^{\rho}\right]^{1/\rho}$$

where $\rho = (\sigma - 1)/\sigma$ and $\sigma$ is the inter-sector substitution elasticity. With $\sigma \to \infty$ (perfect substitutes), this collapses to a weighted sum. With $\sigma \to 0$ (Leontief), sectors are used in fixed proportions.

**Pros**: Theoretically grounded — the KLEM aggregate emerges from micro-level sector demands, and the $\alpha_s$ calibrate to match base-year shares. Price response is consistent across layers.

**Cons**: Requires calibrating $\alpha_s$ and choosing $\sigma$. Adds a new CES function and its Jacobian to the solver. Moderate implementation effort (~50 lines + tests).

---

**Recommendation**: Implement **Option 1** now (one line, zero risk). Plan **Option 2** as the clean target for Phase 3, since it eliminates the conceptual duplication entirely.

## Layer 3: Nested Carrier Choice — Structure

Within each sector, energy carriers compete via a **recursive tree** of preference-factor
logit nodes with stock turnover at every level. The tree has two nesting levels:

```
Sector (root node, k=0.05, τ=50-60y) — structural subsector split
  ├── Subsector A (leaf node, k=0.3, τ=sector-specific) — carrier competition
  │   ├── Coal
  │   ├── Refined liquids
  │   ├── Gas
  │   ├── Electricity
  │   ├── Biomass
  │   └── Hydrogen
  └── Subsector B (same structure)
```

The root node uses **low sensitivity** ($k = 0.05$) and **long turnover** ($\tau = 50\text{–}60$ y)
to represent slow-moving structural change (e.g., the residential/commercial split).
The leaf nodes use **high sensitivity** ($k = 0.3$) and **sector-specific turnover** to
represent actual fuel-switching decisions.

## Layer 3: Preference-Factor Logit + Stock Turnover

At each node, the **target share** for child $i$ is determined by the MERGE-style
preference-factor logit:

$$s_i^{\text{target}} = \frac{\exp\!\bigl(-k\,(C_i + P_i)\bigr)}{\displaystyle\sum_j \exp\!\bigl(-k\,(C_j + P_j)\bigr)}$$

where:
- $C_i$ = levelized cost of child $i$ ($/GJ). For branch children, this is the share-weighted average cost of the subtree.
- $P_i$ = **preference factor** ($/GJ equivalent). Calibrated at the base year so that the logit reproduces observed shares. $P_i > 0$ penalizes (disliked), $P_i < 0$ rewards (preferred).
- $k$ = scale parameter controlling cost-sensitivity.

**Calibration** (at base year): Given observed shares $\bar{s}_i$ and costs $\bar{C}_i$, pick the reference technology $r = \arg\max_i \bar{s}_i$ and set $P_r = 0$. Then:

$$P_i = (\bar{C}_r - \bar{C}_i) - \frac{\ln(\bar{s}_i / \bar{s}_r)}{k}$$

**Stock turnover** then blends the target share with the existing fleet:

$$s_i(t+\Delta t) = s_i(t) + \frac{\Delta t}{\tau} \cdot \bigl(s_i^{\text{target}} - s_i(t)\bigr)$$

with $\Delta t = 5$ years and $\tau$ the sector-specific turnover time.

### CHT: Still too rigid? I think we could use more paremeters? like at least we can add gcam-style share-weight to model the phase-in and -out of technologies?

### CHT: I don't get why the sup-sector like heavy industry has its own tau? I mean sub technologies have lifetime, totally intuitive but sup sector has lifetime?

#### Response: Share-Weights vs. Preference Factors, and Why Subsectors Have τ

**On adding GCAM-style share-weights**: GCAM's share-weights $\alpha_i$ and GHIM's preference factors $P_i$ serve the same role — they calibrate the logit to reproduce observed base-year shares. The difference is functional form:

| | GCAM share-weight $\alpha_i$ | GHIM preference $P_i$ |
|---|---|---|
| Form | Multiplicative: $\alpha_i \cdot C_i^\beta$ | Additive: $\exp(-k(C_i + P_i))$ |
| Unit | Dimensionless (normalized) | $/GJ equivalent |
| Interpretation | Hard to read off directly | Clear: "$P_i = +3$ means technology $i$ carries a \$3/GJ non-cost penalty" |
| Phase-in/out | Exogenous $\alpha_i(t)$ trajectories | Time-varying $P_i(t)$ via decay |

Adding $\alpha_i$ on top of $P_i$ would over-parameterize — both absorb the same calibration residual. We'd have two free parameters per technology doing the same job.

**But CHT's real point is about phase-in/out**: The concern is that fixed $P_i$ can't represent a new technology "phasing in" beyond what cost changes alone allow. This is exactly what **time-varying $P_i$** (already proposed later in this notebook) addresses. The proposed per-technology decay rates $d_i$ are functionally equivalent to GCAM's exogenous share-weight trajectories — they just decay toward cost-purity rather than following an arbitrary trajectory.

**More flexible alternative for explicit phase-in**: Instead of exponential decay, allow **exogenous $P_i(t)$ trajectories** that the user specifies per scenario. For example:

| Year | $P_{\text{hydrogen}}$ | Interpretation |
|------|----------------------|----------------|
| 2025 | +5.0 $/GJ | No infrastructure, high barriers |
| 2030 | +3.0 $/GJ | Early pipelines, some stations |
| 2040 | +1.0 $/GJ | Infrastructure maturing |
| 2050 | 0.0 $/GJ | Full infrastructure, pure cost competition |

This is more flexible than exponential decay and closer to how GCAM modelers actually use share-weight schedules in practice. It could be loaded from the scenario JSON files we already have.

---

**On subsector τ (e.g., heavy industry τ=50y)**: There are two distinct meanings of τ at different tree levels:

- At **leaf nodes** (fuel competition within heavy industry), τ represents **physical capital lifetime** — how fast boilers, furnaces, and kilns are replaced. A coal-fired blast furnace lasts 30 years; that's the turnover time. This is intuitive.

- At the **root node** (heavy vs. light industry split), τ does **not** represent equipment lifetime. It represents **structural inertia in the economy** — the fact that industrial composition changes slowly. Steel mills don't become software companies in 5 years. The heavy/light split reflects physical infrastructure, workforce skills, supply chains, and comparative advantage, all of which evolve on multi-decade timescales.

A long τ=50y at the root prevents the model from rapidly shifting all industry to "light" just because electricity got cheap. Without it, a large cost difference between heavy and light subsectors could cause an implausible structural shift in one period.

**Honest assessment**: This is a modeling convenience, not physics. There's no "lifetime" of industrial structure in the same sense as equipment lifetime. GCAM handles this differently — the heavy/light split is largely **exogenous** (driven by socioeconomic assumptions, not endogenous cost competition). We could alternatively:

1. Make root-level shares **exogenous** from SSP structural change assumptions, removing τ from root nodes entirely
2. Use a very low $k$ at root level (already $k=0.05$ vs $k=0.3$ at leaves) and drop τ, so the split barely moves endogenously

Option 1 is probably cleaner if we have good structural change data from the SSPs. Worth revisiting when we integrate more detailed SSP sectoral assumptions.

## Layer 3: Full Demand Trees

### Industry ($\varepsilon = 0.6$)
```
industry (k=0.05, τ=50y)
├── heavy (k=0.3, τ=30y) — 45%
│   ├── coal           35%
│   ├── gas             25%
│   ├── electricity     15%
│   ├── refined liquids 10%
│   ├── biomass         10%
│   └── hydrogen         5%
├── light (k=0.3, τ=30y) — 45%
│   ├── electricity     40%
│   ├── gas             25%
│   ├── refined liquids 15%
│   ├── coal            10%
│   ├── biomass          8%
│   └── hydrogen         2%
└── data_centers (k=0.3, τ=7y) — 10%
    └── electricity    100%
```

### Buildings ($\varepsilon = 0.5$)
```
buildings (k=0.05, τ=60y)
├── residential (k=0.3, τ=50y) — 55%
│   ├── electricity     35%
│   ├── gas             30%
│   ├── biomass         18%
│   ├── refined liquids 12%
│   ├── coal             4%
│   └── hydrogen         1%
└── commercial (k=0.3, τ=50y) — 45%
    ├── electricity     50%
    ├── gas             30%
    ├── refined liquids  8%
    ├── biomass          8%
    ├── coal             2%
    └── hydrogen         2%
```

### Transport ($\varepsilon = 0.7$)
```
transport (k=0.05, τ=50y)
├── passenger (k=0.3, τ=15y) — 60%
│   ├── refined liquids 87%
│   ├── electricity      5%
│   ├── gas              4%
│   ├── hydrogen         2%
│   └── biomass          2%
└── freight (k=0.3, τ=15y) — 40%
    ├── refined liquids 95%
    ├── gas              2%
    ├── electricity      1%
    ├── hydrogen         1%
    └── biomass          1%
```

## Layer 4: Transformation Sectors — LCOE and Technology Competition

The electricity sector uses the same preference-factor logit among 8 generation technologies.
Each technology's cost is its **levelized cost of energy** (LCOE):

$$\text{LCOE}_i = \frac{\text{CapCost}_i \cdot \text{CRF}(r, L_i)}{\text{CF}_i \cdot 8760 \cdot 3.6 \times 10^{-3}} + \frac{\text{OM}_{\text{fixed},i}}{\text{CF}_i \cdot 8760 \cdot 3.6 \times 10^{-3}} + \frac{P_{\text{fuel},i}}{\eta_i} + \text{OM}_{\text{var},i}$$

where:

| Symbol | Meaning |
|--------|---------|
| $\text{CRF}(r, L)$ | Capital recovery factor: $\frac{r(1+r)^L}{(1+r)^L - 1}$, with $r = 0.05$ |
| $\text{CF}_i$ | Capacity factor |
| $\eta_i$ | Thermal efficiency (output/input) |
| $P_{\text{fuel},i}$ | Fuel price ($/GJ) |

**Current technology parameters** (2020 base year):

| Technology | CapCost ($/kW) | CF | $\eta$ | Lifetime | Fuel |
|-----------|---------------|-----|--------|----------|------|
| Coal | 1500 | 0.75 | 0.39 | 40y | coal |
| Gas CC | 900 | 0.60 | 0.55 | 30y | gas |
| Nuclear | 5500 | 0.90 | 0.33 | 60y | nuclear |
| Hydro | 2500 | 0.45 | 1.0 | 80y | — |
| Wind | 1200 | 0.35 | 1.0 | 25y | — |
| Solar | 900 | 0.22 | 1.0 | 30y | — |
| Biomass | 2500 | 0.70 | 0.35 | 30y | biomass |
| Oil | 800 | 0.30 | 0.37 | 30y | refined liquids |

The electricity sector logit uses $k = 0.3$ and $\tau = 40$ years. Hydrogen uses the same
framework with SMR ($\eta = 0.72$) and electrolysis ($\eta = 0.70$).

### CHT: Is this LCOE cost structure applies only to Transformation sectors? Why? Cuz this is not a special cost structure. For example the passenger liquid cars could have exactly the same strucure and learning-by-doing effect?

#### Response: Why Not LCOE for Demand Side? (CHT Is Right — It's a Gap)

**CHT is correct**: There's nothing structurally preventing demand-side LCOE. Passenger vehicles *do* have capital costs (BEV ~\$40k vs ICE ~\$25k), efficiency differences (BEV ~3× ICE in useful energy per GJ), capacity factors (utilization hours/year), and learning curves (battery pack \$/kWh declining ~15% per capacity doubling). The same LCOE framework applies:

$$\text{LCOE}_{\text{BEV}} = \frac{\text{VehicleCost} \cdot \text{CRF}(r, L)}{\text{AnnualDriving} / \eta_{\text{BEV}}} + \frac{P_{\text{electricity}}}{\eta_{\text{BEV}}} + \text{Maintenance}$$

**Why GHIM doesn't do it currently** — three practical reasons:

1. **Data granularity**: Supply-side technology costs are well-documented (NREL ATB, IEA WEO, IRENA). Demand-side costs vary enormously — vehicle costs in India vs. Japan, industrial boiler costs for different scales, building heating systems across climates. Calibrating demand-side LCOE across 10 regions requires ~6× more technology parameters than we currently have.

2. **Architecture**: `DemandLeaf` is currently a simple 2-field dataclass (carrier name + share). Promoting it to full `Technology` objects (with capital cost, efficiency, lifetime, learning rate, capacity factor) requires refactoring calibration, the solver's price iteration, and the reporting pipeline.

3. **Double-counting risk**: Battery learning appears in both supply-side (grid storage, electrolysis) and demand-side (BEV). If both sectors drive down battery costs independently, total learning could be over-counted. Need a shared learning pool (like WITCH's two-factor learning) to handle cross-sector spillovers.

**What other IAMs do**:

| Model | Demand-side technology detail |
|-------|------------------------------|
| **GCAM** | Full technology nesting in transport (BEV/ICE/FCEV/hybrid), buildings (heat pump/furnace/resistance), industry (electric arc/blast furnace). 3–4 nesting levels. |
| **REMIND** | Detailed transport (BEV/ICE/FCEV with vintage tracking). Buildings and industry at fuel level. |
| **WITCH** | Fuel-level demand (like GHIM). No explicit demand-side technology competition. |
| **MESSAGE** | Full technology detail across all sectors via LP optimization. |

GHIM is currently at the WITCH level of demand-side detail. Moving to GCAM-level would be a major Phase 3 feature.

**Recommended path forward**:

1. **Pilot with transport** (highest impact): Add optional `Technology` objects to transport `DemandLeaf` nodes. Model BEV vs. ICE vs. FCEV competition using the same LCOE + logit framework as electricity. Transport is the best pilot because vehicle costs and efficiencies are well-documented and the BEV transition is the most policy-relevant demand-side shift.

2. **Keep buildings and industry at fuel level** for now — the cost differences between electric heat pumps and gas furnaces are more region-specific and harder to parameterize globally.

3. **Create a separate design notebook** for demand-side technology representation before implementing — the calibration and data requirements are substantial enough to warrant their own design discussion.

## Layer 4: Learning-by-Doing (WITCH-style)

Technologies with positive learning rates see capital cost decline as cumulative deployment grows:

$$\text{CapCost}_i(t) = \text{CapCost}_{i,0} \cdot \left(\frac{Q_{\text{cum},i}(t)}{Q_{i,0}}\right)^{-\beta_i}$$

where $\beta_i = -\ln(1 - \text{LR}_i) / \ln 2$ is the learning exponent and $\text{LR}_i$ is the
learning rate (fractional cost reduction per capacity doubling).

| Technology | Learning rate | Cost floor |
|-----------|--------------|------------|
| Solar | 20% | 20% of initial |
| Wind | 12% | 20% of initial |
| Electrolysis | 15% | 20% of initial |
| Biomass | 5% | 20% of initial |
| Nuclear | 3% | 20% of initial |
| Coal, Gas, Hydro, Oil | 0% | — |

This creates a positive feedback loop: higher deployment → lower cost → higher share → more deployment.
The cost floor at 20% of initial prevents unrealistic cost collapse.

## Layer 5: Price Equilibrium — Damped Fixed-Point Iteration

The solver iterates on three endogenous prices — electricity, refined liquids, hydrogen — until
supply costs converge to demand prices:

$$P_k^{(n+1)} = P_k^{(n)} + \lambda \cdot \bigl(\text{LCOE}_k^{(n)} - P_k^{(n)}\bigr)$$

with damping factor $\lambda = 0.5$, tolerance $\epsilon = 10^{-3}$ (relative), and up to 100 iterations.

**Within each iteration**:
1. Energy price index → total demand (Layer 1)
2. GDP-scaled sector demands → carrier demands via nested logit trees (Layer 2–3)
3. Electricity demand → generation by tech → weighted-average cost → new electricity price (Layer 4)
4. Refined liquids demand → refining cost → new liquids price
5. Hydrogen demand → H₂ tech competition → new hydrogen price
6. Check convergence → damped update

Primary fuel prices (coal, oil, gas) are either exogenous or set by the trade module via
global market clearing (bisection with grade-based supply curves).

## Layer 6: GDP Feedback — Net Output, Investment, Capital Accumulation

Energy costs feed back to GDP through the capital accumulation channel:

$$Y_{\text{net}} = Y_{\text{gross}} - E \cdot \bar{P}$$

$$I = \min\bigl(s \cdot Y_{\text{net}},\; \bar{I} \cdot K\bigr)$$

$$K(t + \Delta t) = (1 - \delta)^{\Delta t} \cdot K(t) + I \cdot \Delta t$$

| Symbol | Meaning | Value |
|--------|---------|-------|
| $s$ | Savings rate | 0.22 |
| $\bar{I}$ | Investment cap (max I/K ratio) | 0.10 |
| $\delta$ | Annual depreciation rate | 0.05 |
| $K/Y$ | Base-year capital-output ratio | 3.0 |

This creates a negative feedback loop: high energy costs → lower net output → lower investment →
lower capital → lower future GDP → lower energy demand. The floor $Y_{\text{net}} \geq 0.01 \cdot Y_{\text{gross}}$
prevents economic collapse.

---
# Design Discussion: Logit Formulation

## Current GHIM Logit — Properties

The MERGE-style preference-factor logit:

$$s_i = \frac{\exp\bigl(-k(C_i + P_i)\bigr)}{\sum_j \exp\bigl(-k(C_j + P_j)\bigr)}$$

**Properties**:

1. **Constant semi-elasticity**: $\frac{\partial \ln(s_i/s_j)}{\partial C_i} = -k$, regardless of cost level. A \$1/GJ cost change always shifts log-share-ratio by $k$.

2. **IIA** (Independence of Irrelevant Alternatives): The ratio $s_i / s_j$ depends only on $C_i, C_j, P_i, P_j$ — adding or removing a third option doesn't change the pairwise ratio.

3. **Additive cost+preference**: Cost and preference enter symmetrically. A preference penalty of $+2$ $/GJ is indistinguishable from a cost increase of $+2$ $/GJ.

4. **Calibration via $P_i$**: Preference factors absorb all non-cost factors (reliability, intermittency, permitting difficulty, public acceptance) into a single $/GJ-equivalent adder. This makes the calibration transparent: you can read off how much the model "likes" or "dislikes" each technology in cost-equivalent terms.

5. **Scale parameter $k$**: With $k = 0.3$ $/GJ$^{-1}$, a \$3/GJ cost difference (roughly coal vs. gas) produces an $e^{-0.3 \times 3} \approx 0.41 \times$ share multiplier. This means a 59% share reduction — strong but not winner-take-all.

## GCAM Logit — Relative Cost Formulation

GCAM uses a **relative-cost** (power-law) logit:

$$s_i = \frac{\alpha_i \cdot C_i^{\beta}}{\sum_j \alpha_j \cdot C_j^{\beta}}$$

with $\beta < 0$ (typically $-3$ to $-6$) and calibrated share weights $\alpha_i$.

**Properties**:

1. **Constant elasticity**: $\frac{\partial \ln(s_i/s_j)}{\partial \ln C_i} = \beta$. A 10% cost change always produces the same *percentage* share-ratio change.

2. **IIA**: Same as GHIM — nested logit is needed to break it.

3. **Multiplicative cost response**: Proportional cost changes matter, not absolute. This naturally scales with cost level — a \$1/GJ change matters more for cheap coal than expensive nuclear.

4. **Calibration via $\alpha_i$**: Share weights are pure multipliers. Their values are hard to interpret directly (they depend on cost units). GCAM normalizes the largest $\alpha$ to 1.

5. **No explicit preference interpretation**: The $\alpha_i$ fuse calibration residuals and non-cost factors into a single multiplier without a clear physical dimension.

## Side-by-Side Comparison: Concrete Example

**Setup**: Coal at \$3/GJ vs Solar at \$8/GJ (both at 50% share in base year).

After solar cost falls to \$5/GJ (coal unchanged):

| | GHIM ($k = 0.3$) | GCAM ($\beta = -4$) |
|---|---|---|
| **Coal share (before)** | 50% | 50% |
| **Solar share (before)** | 50% | 50% |
| **Solar cost change** | \$8 → \$5/GJ | \$8 → \$5/GJ |
| | | |
| **GHIM calculation** | $\Delta C = -3$, so solar's unnorm changes by $e^{-0.3 \times (-3)} = e^{0.9} \approx 2.46\times$ | — |
| Coal unnorm | $e^{-0.3 \times 3} = e^{-0.9}$ | — |
| Solar unnorm | $e^{-0.3 \times 5} = e^{-1.5}$ | — |
| Ratio solar/coal | $e^{-0.3(5-3)} \cdot e^{0} = e^{0.6} \approx 1.82$ | — |
| **Solar share (after)** | $\approx$ **65%** | — |
| | | |
| **GCAM calculation** | — | Cost ratio change: $(5/8)^{-4} = (0.625)^{-4} \approx 6.55\times$ |
| $\alpha$ values | — | Both calibrated to give 50/50 at base costs |
| Solar/coal unnorm ratio | — | $6.55 \times (3/3)^{-4} \cdot (\alpha_s/\alpha_c)$ |
| Recalibrate: $\alpha_c \cdot 3^{-4} = \alpha_s \cdot 8^{-4}$ | — | So $\alpha_s/\alpha_c = (8/3)^4 \approx 50.5$ |
| New ratio: $50.5 \times (5)^{-4} / (3)^{-4}$ | — | $= 50.5 \times (3/5)^4 = 50.5 \times 0.1296 \approx 6.55$ |
| **Solar share (after)** | — | $6.55/(1+6.55) \approx$ **87%** |
| | | |
| **Key difference** | GHIM: absolute \$3 change → moderate shift | GCAM: 37.5% relative change → dramatic shift |

The GCAM logit is more aggressive because the relative cost change (37.5%) is amplified by the power-law exponent. The GHIM logit responds more moderately to the same physical cost change.

## IIA Problem and Nested Logit

Both formulations suffer from **IIA**: adding a new technology (e.g., offshore wind alongside onshore wind) would steal share equally from all existing technologies, not disproportionately from the closest substitute.

**GHIM's current solution**: Two-level nesting.
- **Root nodes** ($k = 0.05$): Low sensitivity, representing structural splits (heavy/light industry, passenger/freight, residential/commercial). Changes slowly.
- **Leaf nodes** ($k = 0.3$): High sensitivity, representing actual carrier competition within each subsector.

The ratio $k_{\text{root}} / k_{\text{leaf}} = 0.05/0.3 \approx 0.17$ controls how much cross-nest substitution occurs. At this ratio, a new electricity carrier in the "heavy industry" subsector primarily cannibalizes other carriers in "heavy industry" rather than shifting the heavy/light split. This partially addresses IIA without full random-coefficients or nested-logit GEV structure.

**GCAM's solution**: Explicit multi-level nesting (up to 4 levels for electricity: sector → subsector → fuel → technology), each level with its own $\beta$. The nesting parameter $\mu = \beta_{\text{child}} / \beta_{\text{parent}}$ must satisfy $0 < \mu \leq 1$ for consistency with random utility maximization.

## IAM Comparison: Technology Choice Mechanisms

| Model | Choice mechanism | Key parameters | Stock dynamics | Learning |
|-------|-----------------|---------------|----------------|----------|
| **GCAM** | Nested relative-cost logit | $\beta = -3$ to $-6$, share weights $\alpha$ | Vintage tracking (explicit cohorts) | Two-factor (global + regional) |
| **MERGE** | Preference-factor logit (additive) | $k$, preference adders $P$ | Fixed lifetime retirement | Floor-cost learning |
| **WITCH** | CES + experience curves | CES $\sigma$, learning exponents | Capital accumulation | Two-factor: LBD + R&D |
| **REMIND** | CES nested + system integration cost | CES $\sigma$, mark-up costs | Vintage capital (putty-clay) | Endogenous + R&D |
| **MESSAGE** | LP cost minimization + constraints | Cost, constraints, reserve margins | Vintage tracking | Exogenous cost trajectories |
| **GHIM** | MERGE-style pref logit + nesting | $k = 0.3$, $P_i$, 2-level nest | Linear blend ($\tau$) | WITCH-style experience curves |

GHIM is closest to MERGE in spirit (additive preference + cost) but borrows learning from WITCH
and nesting from GCAM. The main gap relative to GCAM is the lack of vintage tracking (discussed
in the stock turnover section below).

## Proposed: Time-Varying Preference Factors

**Current state**: `PREF_DECAY_RATE = 0.02` is defined in `config.py` (line 61) and
`preference_decay()` exists in `logit.py` (line 250), but **neither is wired into
the solver or demand trees**. Preference factors are calibrated once at the base year
and then held constant throughout the projection.

**Proposed formula**:

$$P_i(t) = P_{i,0} \cdot (1 - d_i)^{(t - t_0)/\Delta t}$$

where $d_i$ is a **technology-specific** annual decay rate. As $P_i \to 0$, the logit
converges to pure cost competition.

**Suggested decay rates**:

| Technology/carrier | $d_i$ (annual) | Half-life | Rationale |
|-------------------|---------------|-----------|-----------|
| Solar, wind | 0.03 | ~23 years | Rapidly overcoming integration concerns |
| Hydrogen (demand) | 0.03 | ~23 years | Infrastructure build-out removes barriers |
| Electricity (demand) | 0.02 | ~35 years | Gradual electrification acceptance |
| Nuclear | 0.01 | ~69 years | Persistent public acceptance issues |
| Biomass | 0.02 | ~35 years | Sustainability concerns slowly resolve |
| Coal, gas (demand) | 0.00 | $\infty$ | No inherent non-cost preference decay |
| Refined liquids | 0.01 | ~69 years | Slow infrastructure lock-in erosion |

**Implementation**: In `DemandNode.compute_carrier_demands()` and `ElectricitySector.compute_supply()`,
before computing target shares, apply:
```python
self.pref_factors = preference_decay(self.base_pref_factors, years_from_base, decay_rates)
```

**Effect**: Technologies with large positive $P_i$ (penalized in calibration because they have
high observed shares despite high costs, or low shares despite low costs) will gradually lose
their penalty, allowing cost to dominate. This prevents perpetual lock-in of base-year
non-cost advantages.

## Other Generalization Paths (for future reference)

These are not proposed for immediate implementation but worth noting:

### 1. Hybrid Additive-Multiplicative Logit
Combine GHIM's additive preference with GCAM's relative-cost response:

$$s_i = \frac{\exp\bigl(-k \cdot P_i\bigr) \cdot C_i^{\beta}}{\sum_j \exp\bigl(-k \cdot P_j\bigr) \cdot C_j^{\beta}}$$

This lets $P_i$ handle non-cost factors while $\beta$ provides proportional cost response.
Trade-off: two free parameters per technology instead of one.

### 2. Variable $k$ by Sector
Currently $k$ varies only between root (0.05) and leaf (0.3) nodes. Could make $k$
sector-specific (e.g., transport more cost-sensitive than buildings) or even
income-dependent ($k$ decreasing with GDP per capita, representing lower price sensitivity
in wealthy regions).

### 3. Share-Dependent Preference (Bandwagon Effects)
$$P_i(t) = P_{i,0} \cdot f(s_i(t))$$

where $f$ is a decreasing function of market share — as adoption grows, barriers fall.
This can create tipping-point dynamics similar to S-curve diffusion models. Risk:
positive feedback can cause numerical instability.

---
# Design Discussion: Stock Turnover

## Current Model: Linear Blending

The stock turnover mechanism uses linear interpolation between current stock shares and
logit-determined target shares:

$$s_i(t + \Delta t) = s_i(t) + \frac{\Delta t}{\tau} \cdot \bigl(s_i^{\text{target}}(t) - s_i(t)\bigr)$$

**Properties**:

1. **Exponential convergence**: The gap $|s_i - s_i^{\text{target}}|$ decays as $(1 - \Delta t/\tau)^n$ per period. With $\Delta t = 5$, $\tau = 40$: 12.5% of the gap closed per period, or ~74% closed after 10 periods (50 years).

2. **Symmetric**: Growing and declining technologies adjust at the same rate. A technology going from 5% → 50% moves at the same speed as one going from 50% → 5%.

3. **No vintage tracking**: All capacity of technology $i$ is treated as a homogeneous pool. There's no distinction between a 30-year-old coal plant and a 5-year-old one.

4. **Renormalization**: After blending, shares are clipped to $[0, \infty)$ and renormalized to sum to 1. This ensures feasibility but can distort the intended adjustment rates.

5. **Sector-specific $\tau$**:

| Sector | $\tau$ (years) | $\Delta t / \tau$ | Gap closed per period |
|--------|---------------|-------------------|----------------------|
| Data centers | 7 | 0.71 | 71% |
| Transport | 15 | 0.33 | 33% |
| Hydrogen | 25 | 0.20 | 20% |
| Industry | 30 | 0.17 | 17% |
| Electricity | 40 | 0.125 | 12.5% |
| Buildings | 50 | 0.10 | 10% |

## Problem Analysis

### 1. No Vintage Tracking

The current model cannot represent the following real-world situation: a region built many
coal plants in 2000–2010, these have 40-year lifetimes, so they will retire in 2040–2050
regardless of cost competitiveness. Instead, coal share declines smoothly at rate $\Delta t / \tau$
regardless of when the plants were built.

**Impact**: Underestimates near-term lock-in (new plants don't retire early) while
overestimating long-term inertia (old plants should retire faster than the blending formula suggests).

### 2. Symmetric Dynamics

In reality, it's much easier to *build* new capacity than to *retire* existing capacity early:
- Building new solar requires only financing and permitting
- Retiring a coal plant early means stranding assets, displacing workers, losing reliability margin

Conversely, once a technology becomes dominant, growth can be *faster* (supply chains scale,
experience accumulates, financing improves).

The symmetric blending formula misses both effects.

### 3. No S-Curve Dynamics

Empirical technology adoption follows S-curves (slow start → rapid middle → saturation).
The linear blending formula produces exponential convergence (fast start → gradual asymptote),
which is qualitatively wrong for emerging technologies.

**Combined effect**: The model can't reproduce the "tipping point" behavior observed in real
energy transitions (e.g., solar PV going from <1% to 5% in 15 years, then 5% to 30% in
10 years).

## Proposed: Asymmetric Stock Turnover

Replace the single $\tau$ with separate grow/decline turnover times per technology:

$$s_i(t + \Delta t) = s_i(t) + \frac{\Delta t}{\tau_i^*} \cdot \bigl(s_i^{\text{target}}(t) - s_i(t)\bigr)$$

where:

$$\tau_i^* = \begin{cases} \tau_{i,\text{grow}} & \text{if } s_i^{\text{target}} > s_i(t) \quad \text{(technology growing)} \\ \tau_{i,\text{decline}} & \text{if } s_i^{\text{target}} \leq s_i(t) \quad \text{(technology shrinking)} \end{cases}$$

**Suggested values** (electricity sector):

| Technology | $\tau_{\text{grow}}$ (y) | $\tau_{\text{decline}}$ (y) | Asymmetry ratio | Rationale |
|-----------|------------------------|---------------------------|-----------------|-----------|
| Solar | 15 | 40 | 0.38 | Fast build, slow replace (few stranded assets) |
| Wind | 15 | 40 | 0.38 | Same as solar |
| Gas CC | 20 | 35 | 0.57 | Moderate build, moderate retire |
| Nuclear | 40 | 60 | 0.67 | Very slow build (regulation), very slow retire (long-lived) |
| Coal | 40 | 50 | 0.80 | Slow build, even slower retire (asset stranding) |
| Hydro | 50 | 80 | 0.63 | Very long-lived infrastructure |
| Biomass | 20 | 30 | 0.67 | Moderate both ways |
| Oil | 25 | 30 | 0.83 | Aging fleet, not being replaced |

**Key insight**: $\tau_{\text{grow}} < \tau_{\text{decline}}$ for renewables means they can
scale up faster than incumbents decline. This produces asymmetric transitions — consistent
with historical observations that new technologies deploy faster than old ones retire.

**Implementation**: Modify `apply_stock_turnover()` to accept per-technology $\tau$ arrays
instead of a single scalar. Use element-wise comparison of target vs. current to select
grow or decline rate.

## Alternative: Vintage Tracking (Future Reference)

A more rigorous approach tracks capacity by installation year (vintage):

$$\text{Cap}_{i,v}(t) = \text{Cap}_{i,v}(v) \cdot \mathbb{1}[t - v < L_i]$$

where $v$ is the vintage (installation year) and $L_i$ is the technology lifetime. Total
capacity is $\sum_v \text{Cap}_{i,v}(t)$, and new investment follows the logit:

$$\text{NewCap}_i(t) = s_i^{\text{logit}}(t) \cdot \text{TotalNewInvestment}(t)$$

**Advantages**: Captures retirement waves, age-dependent O&M, heat rate degradation.

**Disadvantages**: Requires tracking $N_{\text{tech}} \times N_{\text{vintage}}$ state variables
per region. With 8 technologies and 31 model periods, that's 248 variables per region
(vs. 8 for the current model). Computational cost and complexity increase substantially.

**Verdict**: Worth implementing in a future phase, but asymmetric turnover captures the
most important behavior (grow/decline asymmetry) at minimal additional complexity.

## Logit × Turnover Interaction

The two mechanisms — logit choice and stock turnover — interact in important ways:

### How They Couple

The logit determines $s_i^{\text{target}}$, which is the share that *new investment* would
achieve in a greenfield world. Stock turnover then blends this target with the installed base.
The actual share $s_i(t)$ lies between the two:

$$s_i^{\text{target}} \leftarrow \text{logit}\bigl(C_i(t), P_i(t)\bigr) \quad \longrightarrow \quad s_i(t) = \text{blend}\bigl(s_i(t-1), s_i^{\text{target}}\bigr)$$

The **effective cost** used in next-period demand estimation depends on *actual* shares
(not targets), creating a feedback: slow turnover means the average cost reflects the old
fleet, which in turn affects how much demand shifts to other carriers.

### Path Dependence

Because shares evolve recursively ($s_i(t)$ depends on $s_i(t-1)$), the model exhibits
**path dependence**: two scenarios with the same 2050 costs but different 2020–2045
trajectories will have different 2050 shares. This is a feature, not a bug — it represents
the physical reality of installed infrastructure.

### Incumbent Bonus (Potential Enhancement)

A natural extension: give the logit preference factor a share-dependent component:

$$P_i^{\text{eff}}(t) = P_i(t) - \gamma \cdot \ln\bigl(s_i(t) / s_{\text{ref}}\bigr)$$

where $\gamma > 0$ rewards technologies with large market share (infrastructure exists,
supply chains are scaled, public familiarity). This creates additional inertia beyond
stock turnover alone, representing the "soft" advantages of incumbency.

**Risk**: Combined with stock turnover, this could create excessive lock-in. Would need
careful calibration to avoid preventing any transition at all.

---
## Summary and Next Steps

### Chosen Directions

1. **Time-varying preference factors**:
   - Wire existing `PREF_DECAY_RATE` into `DemandNode` and `ElectricitySector`
   - Make decay rate technology-specific via a new `PREF_DECAY_RATES: dict[str, float]` in `config.py`
   - Default: 2% annual for most, 3% for solar/wind/hydrogen, 0% for fossil fuels, 1% for nuclear
   - Priority: **HIGH** — this is the most impactful single change for long-run projections

2. **Asymmetric stock turnover**:
   - Modify `apply_stock_turnover()` to accept per-technology `(τ_grow, τ_decline)` tuples
   - Update `DemandNode` and `ElectricitySector` to pass technology-level turnover parameters
   - Default: grow 30–50% faster than decline for renewables; symmetric for fossil fuels
   - Priority: **MEDIUM** — important for transition dynamics, but second-order vs. preference decay

### Not Planned (Parked for Future)

- Hybrid additive-multiplicative logit — adds complexity without clear calibration advantage
- Full vintage tracking — significant refactor, defer to Phase 3
- Incumbent bonus — risk of over-parameterization; revisit after preference decay is tested
- Share-dependent preference — numerical stability concerns; needs careful testing

### Implementation Order

1. Add `PREF_DECAY_RATES` dict to `config.py`
2. Wire `preference_decay()` into `ElectricitySector.compute_supply()` (using `years_from_base`)
3. Wire into `DemandNode.compute_carrier_demands()` (needs `years_from_base` passed down)
4. Add asymmetric $\tau$ to `apply_stock_turnover()` and `Technology` dataclass
5. Update tests for both features
6. Validate against electricity_validation.ipynb: check that share evolution is reasonable